In [1]:
!pip install optuna
!pip install -U kaleido
!pip install matplotlib
!pip install requests

In [2]:
import numpy as np
import pandas as pd
import torch

In [3]:
from sklearn.preprocessing import OrdinalEncoder

class Encoder:

    def __init__(self) -> None:
        self.user_encoder = OrdinalEncoder(dtype=int, handle_unknown='use_encoded_value', unknown_value=-1)
        self.item_encoder = OrdinalEncoder(dtype=int, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, interactions: pd.DataFrame) -> pd.DataFrame:
        self.user_encoder.fit(
            interactions['user_id'].values.reshape(-1, 1)
        )
        self.item_encoder.fit(
            interactions['item_id'].values.reshape(-1, 1)
        )
        return self

    def transform(self, interactions: pd.DataFrame) -> pd.DataFrame:
        interactions = interactions.copy()
        interactions.loc[:, 'user_id'] = self.user_encoder.transform(
            interactions['user_id'].values.reshape(-1, 1)
        ).ravel()
        interactions.loc[:, 'item_id'] = self.item_encoder.transform(
            interactions['item_id'].values.reshape(-1, 1)
        ).ravel()
        if (interactions['user_id'] == -1).any() or (interactions['item_id'] == -1).any():
            print(f'Found {len(interactions[interactions["user_id"] == -1])} unknown users!')
            print(f'Found {len(interactions[interactions["item_id"] == -1])} unknown items!')
        interactions = interactions[
            (interactions['user_id'] != -1) &
            (interactions['item_id'] != -1)
        ]
        return interactions

    def fit_transform(self, interactions: pd.DataFrame) -> pd.DataFrame:
        return self.fit(interactions).transform(interactions)

    def inverse_transform(self, interactions: pd.DataFrame) -> pd.DataFrame:
        interactions.loc[:, 'user_id'] = self.user_encoder.inverse_transform(
            interactions['user_id'].values.reshape(-1, 1)
        ).ravel()
        interactions.loc[:, 'item_id'] = self.item_encoder.inverse_transform(
            interactions['item_id'].values.reshape(-1, 1)
        ).ravel()
        return interactions

In [4]:
from functools import cached_property
from scipy.sparse import csr_matrix, vstack, hstack, diags

class Dataset:
    def __init__(self, train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        self._check_integrity()

    def _check_integrity(self) -> None:
        ordinal_series = [
            self.train_df['user_id'],
            self.train_df['item_id'],
        ]
        for series in ordinal_series:
            assert series.min() == 0
            assert series.max() == series.nunique() - 1

    @cached_property
    def user_cnt(self) -> int:
        return self.train_df['user_id'].nunique()

    @cached_property
    def item_cnt(self) -> int:
        return self.train_df['item_id'].nunique()

    @cached_property
    def interaction_cnt(self) -> int:
        return len(self.train_df)

    @cached_property
    def density(self) -> float:
        return self.interaction_cnt / (self.user_cnt * self.item_cnt)

    @cached_property
    def user_item_matrix(self) -> csr_matrix:
        user_ids = self.train_df['user_id']
        item_ids = self.train_df['item_id']
        data = np.ones_like(user_ids)
        user_item_matrix = csr_matrix((data, (user_ids, item_ids)), shape=(self.user_cnt, self.item_cnt))
        return user_item_matrix

    @cached_property
    def adj_matrix(self) -> csr_matrix:
        return self.user_item_matrix

    @cached_property
    def extended_adj_matrix(self) -> csr_matrix:
        upper_left_zeros = csr_matrix((self.user_cnt, self.user_cnt))
        upper_part = hstack([upper_left_zeros, self.adj_matrix])
        lower_right_zeros = csr_matrix((self.item_cnt, self.item_cnt))
        lower_part = hstack([self.adj_matrix.transpose(), lower_right_zeros])
        extended_adj_matrix = vstack([upper_part, lower_part])
        return extended_adj_matrix

    @cached_property
    def normalized_matrix(self) -> csr_matrix:
        row_sum = np.array(self.extended_adj_matrix.sum(axis=1)).squeeze()
        row_sum[row_sum == 0] = 1.0
        normalizer = diags(row_sum ** -0.5)
        normalized_matrix = normalizer @ self.extended_adj_matrix @ normalizer
        return normalized_matrix

In [5]:
import os

def pick_users_with_exact_n(train_df: pd.DataFrame, n: int, n_users=None, seed: int = 0):
    # train에서 상호작용 수 정확히 n인 유저 리스트만 리턴 (train은 변경 X)
    rng = np.random.default_rng(seed)
    cnt = train_df.groupby("user_id").size()
    users = cnt[cnt == n].index.to_numpy()

    if n_users is not None:
        n_users = min(n_users, len(users))
        users = rng.choice(users, size=n_users, replace=False)

    return users.tolist()

def make_1shot_from_exact_3(train_df: pd.DataFrame, n_users=None, seed: int = 0):
    # train에서 상호작용 수가 '정확히 3'인 유저들 중 일부를 골라 각 유저당 1개만 남기고(2개 제거) train_df를 수정해서 리턴
    
    rng = np.random.default_rng(seed)

    cnt = train_df.groupby("user_id").size()
    users3 = cnt[cnt == 3].index.to_numpy()

    if n_users is not None:
        n_users = min(n_users, len(users3))
        chosen = rng.choice(users3, size=n_users, replace=False)
    else:
        chosen = users3

    chosen = set(chosen.tolist())

    keep_idx = []
    for u, g in train_df.groupby("user_id", sort=False):
        idx = g.index.to_numpy()
        if u in chosen:
            # 3개 중 1개만 남김
            keep_idx.append(rng.choice(idx, size=1, replace=False))
        else:
            keep_idx.append(idx)

    keep_idx = np.concatenate(keep_idx)
    new_train = train_df.loc[keep_idx].copy()

    new_cnt = new_train.groupby("user_id").size()
    target_users = new_cnt[new_cnt == 1].index.tolist()

    return new_train, target_users

def get_dataset(dataset_name: str, exp_cfg: dict | None = None) -> Dataset:

    def is_colab():
        return "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
    if is_colab():
        from google.colab import drive
        drive.mount('/content/drive')
        shns_data_path = '/content/drive/MyDrive/SHNS_data'
    else:
        shns_data_path = './SHNS_data'

    data_path = shns_data_path + '/' + dataset_name

    def load_train_df(path):
        rows = []
        with open(path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                assert len(parts) == 2
                rows.append({'user_id': int(parts[0]), 'item_id': int(parts[1])})
        return pd.DataFrame(rows, columns=['user_id', 'item_id'])

    def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
        num_duplicates = df.duplicated(subset=["user_id", "item_id"]).sum()
        if num_duplicates > 0:
            print(f"Found {num_duplicates} duplicate rows")
            df = df.drop_duplicates(subset=["user_id", "item_id"]).reset_index(drop=True)
        else:
            print("No duplicate rows found")
        return df

    train_df = load_train_df(data_path + '/train.txt')
    valid_df = load_train_df(data_path + '/valid.txt')
    test_df = load_train_df(data_path + '/test.txt')
    train_df = remove_duplicates(train_df)
    valid_df = remove_duplicates(valid_df)
    test_df = remove_duplicates(test_df)
    encoder = Encoder()
    train_df = encoder.fit_transform(train_df)
    valid_df = encoder.transform(valid_df)
    test_df = encoder.transform(test_df)

    target_users = None
    if exp_cfg is not None:
        mode = exp_cfg["mode"]
        seed = exp_cfg.get("seed", 0)
        n_users = exp_cfg.get("n_users", None)

        if mode == "pseudo_1shot_from_3":
            train_df, target_users = make_1shot_from_exact_3(train_df, n_users=n_users, seed=seed)
        elif mode == "natural_5shot":
            target_users = pick_users_with_exact_n(train_df, n=5, n_users=n_users, seed=seed)

        else:
            raise ValueError(f"Unknown exp_cfg mode: {mode}")
        
        print(f"[exp_cfg] mode={mode}, targets={len(target_users)}")

    dataset = Dataset(train_df, valid_df, test_df)

    # Dataset 클래스 바꾸기 싫으면 속성으로만 붙여도 됨
    if target_users is not None:
        dataset.target_users = target_users

    return dataset

In [6]:
from typing import Tuple

class LightGCN(torch.nn.Module):
    def __init__(self, dataset: Dataset, hyperparams: dict) -> None:
        super(LightGCN, self).__init__()
        self.dataset = dataset
        self.hyperparams = hyperparams
        self.user_embeddings = torch.nn.Embedding(
            self.dataset.user_cnt, self.hyperparams['emb_size']
        )
        self.item_embeddings = torch.nn.Embedding(
            self.dataset.item_cnt, self.hyperparams['emb_size']
        )

        train_trues = self.dataset.train_df.groupby('user_id')['item_id'].apply(list)
        self.train_mask = torch.zeros((self.dataset.user_cnt, self.dataset.item_cnt), dtype=torch.bool)
        for user_id, items in train_trues.items():
           self.train_mask[user_id, items] = 1

        # for ANS (+ DENS)
        self.user_gate = torch.nn.Linear(self.hyperparams['emb_size'], self.hyperparams['emb_size'])
        self.item_gate = torch.nn.Linear(self.hyperparams['emb_size'], self.hyperparams['emb_size'])
        self.margin_model = torch.nn.Linear(self.hyperparams['emb_size'], 1)
        self.pos_gate = torch.nn.Linear(self.hyperparams['emb_size'], self.hyperparams['emb_size'])
        self.neg_gate = torch.nn.Linear(self.hyperparams['emb_size'], self.hyperparams['emb_size'])

        torch.nn.init.xavier_uniform_(self.user_embeddings.weight)
        torch.nn.init.xavier_uniform_(self.item_embeddings.weight)

        self.aggregator = self.get_aggregator()

    def forward(self, user_indices: torch.Tensor, item_indices: torch.Tensor) -> torch.Tensor:
        user_embeddings, item_embeddings = self.get_embeddings()
        user_embeddings = user_embeddings[user_indices]
        item_embeddings = item_embeddings[item_indices]
        return torch.sum(user_embeddings * item_embeddings, dim=1)

    def get_embeddings(self, aggregate=True) -> Tuple[torch.Tensor, torch.Tensor]:
        embeddings = []
        full_embedding = torch.cat([self.user_embeddings.weight, self.item_embeddings.weight], dim=0)
        embeddings.append(full_embedding)
        for _ in range(self.hyperparams['layer_num']):
            full_embedding = torch.sparse.mm(self.aggregator, full_embedding)
            embeddings.append(full_embedding)
        final_embedding = torch.stack(embeddings, dim=0)
        if aggregate:
            final_embedding = torch.mean(final_embedding, dim=0)
        else:
            final_embedding = torch.permute(final_embedding, (1, 0, 2))
        final_user_embedding, final_item_embedding = torch.split(
            final_embedding, [self.dataset.user_cnt, self.dataset.item_cnt],
        )
        return final_user_embedding, final_item_embedding

    def get_aggregator(self) -> torch.Tensor:
        coo = self.dataset.normalized_matrix.tocoo()
        indices = torch.tensor(np.array([coo.row, coo.col]), dtype=torch.long)
        values = torch.tensor(coo.data, dtype=torch.float32)
        aggregator = torch.sparse_coo_tensor(indices, values, size=coo.shape)
        return aggregator
    
    def get_topk(self, k: int) -> torch.Tensor:
        user_embeddings, item_embeddings = self.get_embeddings()

        n_users = user_embeddings.size(0)
        batch_size = 512

        item_emb_T = item_embeddings.T
        topk_list = []

        for start in range(0, n_users, batch_size):
            end = min(start + batch_size, n_users)
            batch_user_emb = user_embeddings[start:end]
            scores = batch_user_emb @ item_emb_T
            mask_batch = self.train_mask[start:end]
            if mask_batch.device != scores.device:
                mask_batch = mask_batch.to(scores.device)
            scores = scores.masked_fill(mask_batch, float("-inf"))
            batch_topk = torch.topk(scores, k=k, dim=1).indices
            topk_list.append(batch_topk)

        topk = torch.cat(topk_list, dim=0)
        return topk

    def to(self, device: torch.device):
        super(LightGCN, self).to(device)
        self.train_mask = self.train_mask.to(device)
        self.aggregator = self.aggregator.to(device)
        return self

In [7]:
import numpy as np
import torch
from scipy.sparse import csr_matrix

class Sampler:
    def __init__(self, model, dataset, hyperparams: dict) -> None:
        self.model = model
        self.dataset = dataset
        self.hyperparams = hyperparams
        self.device = "cuda"

        self.U = int(dataset.user_cnt)
        self.I = int(dataset.item_cnt)
        self.C = int(hyperparams["cand_size"])
        self.max_round = int(hyperparams.get("sampler_max_round", 50))

        adj: csr_matrix = dataset.user_item_matrix.tocsr()
        indptr = adj.indptr.astype(np.int64)
        indices = adj.indices.astype(np.int64)

        users_edge = np.repeat(np.arange(self.U, dtype=np.int64), np.diff(indptr))
        pos_edge = indices.astype(np.int64)

        self.users_edge = torch.from_numpy(users_edge).long()
        self.pos_edge = torch.from_numpy(pos_edge).long()
        self.K = int(self.pos_edge.numel())

        key_pos = users_edge * self.I + pos_edge
        self.key_pos_sorted = torch.from_numpy(np.sort(key_pos)).to(self.device)

    def set_cand_size(self, C: int):
        self.C = int(C)

    @torch.no_grad()
    def _exists_in_pos(self, keys_query: torch.Tensor) -> torch.Tensor:
        flat = keys_query.reshape(-1)
        idx = torch.searchsorted(self.key_pos_sorted, flat)
        idx = idx.clamp(0, self.key_pos_sorted.numel() - 1)
        hit = self.key_pos_sorted[idx] == flat
        return hit.view_as(keys_query)

    @torch.no_grad()
    def get_samples(self, batch_idx: torch.Tensor) -> torch.Tensor:
        if batch_idx.device.type != "cpu":
            batch_idx = batch_idx.detach().cpu()

        users = self.users_edge[batch_idx].to(self.device, non_blocking=True)
        pos = self.pos_edge[batch_idx].to(self.device, non_blocking=True)

        B = int(users.numel())
        neg = torch.randint(0, self.I, (B, self.C), device=self.device, dtype=torch.long)

        def conflict_mask(neg_):
            same_pos = neg_.eq(pos.unsqueeze(1))
            key = users.unsqueeze(1).to(torch.int64) * self.I + neg_.to(torch.int64)
            in_pos = self._exists_in_pos(key)
            return same_pos | in_pos

        conflict = conflict_mask(neg)
        r = 0
        while conflict.any():
            r += 1
            if r > self.max_round:
                break
            m = conflict
            neg[m] = torch.randint(0, self.I, (int(m.sum().item()),), device=self.device)
            conflict = conflict_mask(neg)

        return torch.cat([users.view(B, 1), pos.view(B, 1), neg], dim=1)


In [8]:
import math
from typing import List

def get_recall(pred: List[List[int]], true: List[List[int]]) -> float:
    total_recall = 0.0
    valid_cnt = 0
    for p, t in zip(pred, true):
        if len(t) == 0:
            continue
        hit = len(set(p) & set(t))
        total_recall += hit / len(t)
        valid_cnt += 1
    return total_recall / valid_cnt if valid_cnt > 0 else -1

def get_ndcg(pred: List[List[int]], true: List[List[int]]) -> float:
    total_ndcg = 0.0
    valid_cnt = 0
    for p, t in zip(pred, true):
        if len(t) == 0:
            continue
        dcg = 0.0
        for idx, item in enumerate(p):
            if item in t:
                dcg += 1 / math.log2(idx + 2)
        ideal_len = min(len(t), len(p))
        idcg = sum(1 / math.log2(i + 2) for i in range(ideal_len))
        total_ndcg += dcg / idcg if idcg > 0 else 0.0
        valid_cnt += 1
    return total_ndcg / valid_cnt if valid_cnt > 0 else -1

In [9]:
import matplotlib.pyplot as plt
import math

def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.log(1 + torch.exp(neg_scores - pos_scores)))

def model_reg_loss(user_embs: torch.Tensor, pos_embs: torch.Tensor, neg_embs: torch.Tensor) -> torch.Tensor:
    reg_loss = torch.norm(user_embs) ** 2 + torch.norm(pos_embs) ** 2 + torch.norm(neg_embs) ** 2
    reg_loss /= len(user_embs)
    return reg_loss

class Trainer:
    def __init__(self, dataset: Dataset, eval_users=None):
        self.dataset = dataset
        self.eval_users = eval_users
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # self.device = torch.device('cpu')
        valid_trues = self.dataset.val_df.groupby('user_id')['item_id'].apply(list).to_dict()
        self.valid_trues = [valid_trues[i] if i in valid_trues else [] for i in range(self.dataset.user_cnt)]
        test_trues = self.dataset.test_df.groupby('user_id')['item_id'].apply(list).to_dict()
        self.test_trues = [test_trues[i] if i in test_trues else [] for i in range(self.dataset.user_cnt)]

    def _get_alpha(self, smooth_epoch: float, B: int, device: torch.device) -> torch.Tensor:
        hp = self.hyperparams

        if 'alpha' in hp:
            alpha = torch.as_tensor(hp['alpha'], device=device, dtype=torch.float32)
        elif 'alpha_step' in hp:
            alpha_start = float(hp.get('alpha_start', 0.001))
            alpha_max = float(hp.get('alpha_max', 0.5))
            alpha_step = float(hp['alpha_step'])

            alpha = alpha_start + alpha_step * float(smooth_epoch)
            alpha = max(0.0, min(alpha, alpha_max))
            alpha = torch.as_tensor(alpha, device=device, dtype=torch.float32)
        else:
            raise ValueError("alpha is not set: provide 'alpha' or 'alpha_step' (+ optional alpha_start/alpha_max))")
        
        if alpha.ndim == 0:
            alpha = alpha.expand(B)
        return alpha

    def dns_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        user_embeddings, item_embeddings = model.get_embeddings()
        user_neg = user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]
        user_neg = torch.sum(user_neg, dim=-1)
        neg_indices = torch.argmax(user_neg, dim=1).detach()
        neg = neg_cand[torch.arange(neg_cand.size(0)), neg_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]
        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * neg_embeddings, dim=1)
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def dns_mn_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        B, C = neg_cand.size()
        user_embeddings, item_embeddings = model.get_embeddings()
        user_neg = user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]
        user_neg = torch.sum(user_neg, dim=-1)

        m = int(self.hyperparams['m_ratio'] * (C - 1)) + 1
        if m == 1:
            neg_rank = torch.zeros((B,), device=self.device, dtype=torch.long)
        else:
            neg_rank = torch.randint(0, m, (B,), device=self.device)
        neg_cand_indices = torch.argsort(user_neg, dim=1, descending=True)[torch.arange(B), neg_rank]
        neg = neg_cand[torch.arange(neg_cand.size(0)), neg_cand_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]
        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * neg_embeddings, dim=1)
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def shns_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:

        B, C = neg_cand.size()
        def stochastic_round(x: torch.Tensor) -> torch.Tensor:
            lower = torch.floor(x)
            prob_up = x - lower
            rand = torch.rand_like(x)
            rounded = torch.where(rand < prob_up, lower + 1, lower)
            return rounded.to(torch.long)

        user_embeddings, item_embeddings = model.get_embeddings()
        user_neg = user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]
        user_neg = torch.sum(user_neg, dim=-1)

        if 'alpha' in self.hyperparams:
            alpha = self.hyperparams['alpha']
        elif 'alpha_step' in self.hyperparams:
            alpha = self.hyperparams['alpha_step'] * self.hyperparams['smooth_epoch']

        neg_rank = alpha * (C - 1)
        neg_rank = torch.full((B,), neg_rank, dtype=torch.float32)
        neg_rank = torch.clamp(neg_rank, min=0, max=(C - 1))
        neg_rank = stochastic_round(neg_rank)

        neg_cand_indices = torch.argsort(user_neg, dim=1, descending=True)[torch.arange(B), neg_rank]
        neg = neg_cand[torch.arange(B), neg_cand_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]

        pos_weight = torch.exp(user_embeddings[users] * pos_embeddings)
        neg_weight = torch.exp(user_embeddings[users] * neg_embeddings)
        interp_weight = neg_weight / (neg_weight + self.hyperparams['beta'] * pos_weight)
        mixed_neg_embeddings = neg_embeddings * interp_weight + pos_embeddings * (1 - interp_weight)

        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * mixed_neg_embeddings, dim=1)
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def ahns_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        user_embeddings, item_embeddings = model.get_embeddings()
        target_diffs = self.hyperparams['beta'] * (
            (user_embeddings[users] * item_embeddings[pos]).sum(dim=1) + self.hyperparams['alpha']
        ) ** -1

        diffs = (user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]).sum(dim=-1)
        ratings = torch.abs(target_diffs.unsqueeze(-1) - diffs)
        neg_indices = torch.argmin(ratings, dim=1)
        neg = neg_cand[torch.arange(neg_cand.size(0)), neg_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]
        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * neg_embeddings, dim=1)
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def pure_shns_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        B, C = neg_cand.size()
        def stochastic_round(x: torch.Tensor) -> torch.Tensor:
            lower = torch.floor(x)
            prob_up = x - lower
            rand = torch.rand_like(x)
            rounded = torch.where(rand < prob_up, lower + 1, lower)
            return rounded.to(torch.long)

        user_embeddings, item_embeddings = model.get_embeddings()
        user_neg = (user_embeddings[users].unsqueeze(1) * item_embeddings[neg_cand]).sum(dim=-1)

        alpha = self._get_alpha(
            smooth_epoch=self.hyperparams['smooth_epoch'],
            B=B,
            device=users.device
        )

        max_rank = torch.full((B,), C - 1, device=users.device, dtype=torch.float32)
        neg_rank = torch.clamp(alpha * max_rank, min=0, max=(C - 1))
        neg_rank = torch.minimum(neg_rank, max_rank)
        neg_rank = stochastic_round(neg_rank).long()

        ar = torch.arange(B, device=users.device)
        sorted_idx = torch.argsort(user_neg, dim=1, descending=True)
        neg_cand_indices = sorted_idx[ar, neg_rank]
        neg = neg_cand[ar, neg_cand_indices].detach()

        pos_embeddings = item_embeddings[pos]
        neg_embeddings = item_embeddings[neg]
        pos_scores = torch.sum(user_embeddings[users] * pos_embeddings, dim=1)
        neg_scores = torch.sum(user_embeddings[users] * neg_embeddings, dim=1)
        
        loss = bpr_loss(pos_scores, neg_scores)
        reg_loss = model_reg_loss(model.user_embeddings(users), model.item_embeddings(pos), model.item_embeddings(neg))
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def dins_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        def pooling(embeddings):
            return embeddings.mean(dim=1)

        user, pos_item, neg_candidates = users, pos, neg_cand
        user_gcn_emb, item_gcn_emb = model.get_embeddings(aggregate=False)

        # code from https://github.com/Wu-Xi/DINS/blob/main/modules/LightGCN_DINS.py

        batch_size = user.shape[0]
        s_e, p_e = user_gcn_emb[user], item_gcn_emb[pos_item]  # [batch_size, n_hops+1, channel]
        s_e = pooling(s_e).unsqueeze(dim=1)

        """Hard Boundary Definition"""
        n_e = item_gcn_emb[neg_candidates]  # [batch_size, n_negs, n_hops, channel]
        scores = (s_e.unsqueeze(dim=1) * n_e).sum(dim=-1)  # [batch_size, n_negs, n_hops+1]
        indices = torch.max(scores, dim=1)[1].detach()  # torch.Size([2048, 3])
        neg_items_emb_ = n_e.permute([0, 2, 1, 3])  # [batch_size, n_hops+1, n_negs, channel]
        neg_items_embedding_hardest = neg_items_emb_[[[i] for i in range(batch_size)],range(neg_items_emb_.shape[1]), indices, :]   #   [batch_size, n_hops+1, channel]

        """Dimension Independent Mixup"""
        neg_scores = torch.exp(s_e *neg_items_embedding_hardest)  # [batch_size, n_hops, channel]
        total_sum = self.hyperparams['beta'] * torch.exp ((s_e * p_e))+neg_scores   # [batch_size, n_hops, channel]
        neg_weight = neg_scores/total_sum     # [batch_size, n_hops, channel]
        pos_weight = 1-neg_weight   # [batch_size, n_hops, channel]

        n_e_ =  pos_weight * p_e + neg_weight * neg_items_embedding_hardest  # mixing

        neg_gcn_embs = n_e_
        neg_gcn_embs = neg_gcn_embs.unsqueeze(dim=1)
        user_gcn_emb = user_gcn_emb[user]
        pos_gcn_embs = item_gcn_emb[pos_item]

        u_e = pooling(user_gcn_emb)
        pos_e = pooling(pos_gcn_embs)
        neg_e = pooling(neg_gcn_embs.view(-1, neg_gcn_embs.shape[2], neg_gcn_embs.shape[3])).view(batch_size, 1, -1)

        pos_scores = torch.sum(torch.mul(u_e, pos_e), axis=1)
        neg_scores = torch.sum(torch.mul(u_e.unsqueeze(dim=1), neg_e), axis=-1)  # [batch_size, K]
        loss = bpr_loss(pos_scores, neg_scores.squeeze())
        reg_loss = model_reg_loss(user_gcn_emb[:, 0, :], pos_gcn_embs[:,0, :], neg_gcn_embs[:, :, 0, :])
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def ans_loss(self, model: LightGCN, users: torch.Tensor, pos: torch.Tensor, neg_cand: torch.Tensor) -> torch.Tensor:
        user, pos_item, neg_item = users, pos, neg_cand
        user_all_embeddings, item_all_embeddings = model.get_embeddings(aggregate=False)

        # code from https://github.com/Asa9aoTK/ANS-Recbole/blob/main/recbole/model/general_recommender/ans.py
        u_embeddings = user_all_embeddings[user]
        pos_embeddings = item_all_embeddings[pos_item]
        neg_embeddings = item_all_embeddings[neg_item]

        s_e = u_embeddings
        p_e = pos_embeddings
        n_e = neg_embeddings
        batch_size = user.shape[0]

        gate_neg_hard = torch.sigmoid(model.item_gate(n_e) * model.user_gate(s_e).unsqueeze(1))
        n_hard =  n_e * gate_neg_hard
        n_easy =  n_e - n_hard

        p_hard =  p_e.unsqueeze(1) * gate_neg_hard
        p_easy =  p_e.unsqueeze(1) - p_hard

        import torch.nn.functional as F
        distance = torch.mean(F.pairwise_distance(n_hard, p_hard, p=2).squeeze(dim=1))
        temp = torch.norm(torch.mul(p_easy, n_easy),dim=-1)
        orth = torch.mean(torch.sum(temp,axis=-1))

        margin = torch.sigmoid(1/model.margin_model(n_hard * p_hard))

        random_noise = torch.rand(n_easy.shape).to(self.device)
        magnitude = torch.nn.functional.normalize(random_noise, p=2, dim=-1) * margin *0.1
        direction = torch.sign(p_easy - n_easy)
        noise = torch.mul(direction,magnitude)
        n_easy_syth = noise + n_easy
        n_e_ = n_hard + n_easy_syth
        hard_scores = torch.sum(torch.mul(s_e.unsqueeze(dim=1), n_hard), axis=-1)  # [batch_size, K]
        easy_scores = torch.sum(torch.mul(s_e.unsqueeze(dim=1), n_easy), axis=-1)  # [batch_size, K]
        syth_scores = torch.sum(torch.mul(s_e.unsqueeze(dim=1), n_e_), axis=-1)  # [batch_size, K]
        norm_scores = torch.sum(torch.mul(s_e.unsqueeze(dim=1), n_e), axis=-1)  # [batch_size, K]
        sns_loss = torch.mean(torch.log(1 + torch.exp(easy_scores - hard_scores).sum(dim=1)))
        dis_loss = distance + orth
        scores = (s_e.unsqueeze(dim=1) * n_e_).sum(dim=-1)  # [batch_size, n_negs]
        scores_false =  syth_scores - norm_scores

        indices = torch.max(scores + self.hyperparams['epsilon']*scores_false, dim=1)[1].detach()
        neg_items_emb_ = n_e_.permute([0, 2, 1, 3])  # [batch_size, n_hops+1, n_negs, channel]
        # [batch_size, n_hops+1, channel]
        neg_embeddings = neg_items_emb_[[[i] for i in range(batch_size)],range(neg_items_emb_.shape[1]), indices, :]

        # calculate BPR Loss
        pos_scores = torch.mul(u_embeddings, pos_embeddings).sum(dim=1).squeeze(dim=1).sum(dim=-1)
        neg_scores = torch.mul(u_embeddings, neg_embeddings).sum(dim=1).sum(dim=1)
        mf_loss = bpr_loss(pos_scores, neg_scores)

        # calculate BPR Loss
        u_ego_embeddings = model.user_embeddings(user)
        pos_ego_embeddings = model.item_embeddings(pos_item)
        neg_ego_embeddings = model.item_embeddings(neg_item)

        loss = mf_loss + self.hyperparams['gamma'] * (sns_loss + dis_loss)
        reg_loss = model_reg_loss(u_ego_embeddings, pos_ego_embeddings, neg_ego_embeddings)
        loss += reg_loss * self.hyperparams['reg']
        return loss

    def train_model(self, model: LightGCN, sampler: Sampler, hyperparams: dict) -> LightGCN:
        self.hyperparams = hyperparams
        model = model.to(self.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        cand_c = int(hyperparams["cand_size"])

        if hasattr(sampler, "set_cand_size"):
            sampler.set_cand_size(cand_c)
        else:
            sampler.cand_size = cand_c

        self.hyperparams["cand_size"] = cand_c

        idx_loader = torch.utils.data.DataLoader(
            torch.arange(sampler.K),
            batch_size=hyperparams["batch_size"],
            shuffle=True,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True,
            prefetch_factor=4
        )

        n_epochs = int(hyperparams.get("epochs", 1))

        for epoch in range(n_epochs):
            self.hyperparams['epoch'] = epoch

            for step, batch_idx in enumerate(idx_loader):
                smooth_epoch = epoch + (step / len(idx_loader))
                self.hyperparams['smooth_epoch'] = smooth_epoch

                cur_batch = sampler.get_samples(batch_idx)

                optimizer.zero_grad()
                users = cur_batch[:, 0]
                pos = cur_batch[:, 1]
                neg_cand = cur_batch[:, 2:]

                loss = getattr(self, self.hyperparams['method'] + '_loss')(model, users, pos, neg_cand)
                loss.backward()
                optimizer.step()
            return model

    def validate(self, model: LightGCN) -> tuple:
        result = {}

        eval_users = self.eval_users if self.eval_users is not None else range(self.dataset.user_cnt)

        for topk in [10, 20]:
            model.eval()
            with torch.no_grad():
                preds_all = model.get_topk(topk).to('cpu').numpy().tolist()
            model.train()

            preds = [preds_all[u] for u in eval_users]
            trues = [self.valid_trues[u] for u in eval_users]

            cur_recall = get_recall(preds, trues)
            cur_ndcg = get_ndcg(preds, trues)
            result[f'recall_{topk}'] = cur_recall
            result[f'ndcg_{topk}'] = cur_ndcg
        return result

    def test(self, model: LightGCN) -> tuple:
        result = {}

        eval_users = self.eval_users if self.eval_users is not None else range(self.dataset.user_cnt)

        for topk in [10, 20]:
            model.eval()
            with torch.no_grad():
                preds_all = model.get_topk(topk).to('cpu').numpy().tolist()
            model.train()

            preds = [preds_all[u] for u in eval_users]
            trues = [self.test_trues[u] for u in eval_users]

            cur_recall = get_recall(preds, trues)
            cur_ndcg = get_ndcg(preds, trues)
            result[f'recall_{topk}'] = cur_recall
            result[f'ndcg_{topk}'] = cur_ndcg
        return result

In [10]:
import torch

def train(dataset: Dataset, hyperparams: dict, print_result: bool = False):
    trainer = Trainer(dataset, eval_users=None)
    test_model = LightGCN(dataset, hyperparams).to('cuda')

    best_val_result = None
    best_test_result = None
    best_recall = -1e10
    patience = int(hyperparams.get("patience", 10))
    max_epochs = int(hyperparams.get("epochs", 100))

    sampler = Sampler(test_model, dataset, hyperparams)

    for epoch in range(max_epochs):
        hyperparams['epoch'] = epoch
        trainer.train_model(test_model, sampler, hyperparams)
        val_result = trainer.validate(test_model)
        test_result = trainer.test(test_model)

        cur_recall = val_result['recall_20']
        if print_result:
            print(cur_recall)
        if cur_recall > best_recall:
            best_recall = cur_recall
            best_val_result = val_result
            best_test_result = test_result
            patience = int(hyperparams.get("patience", 10))
        else:
            patience -= 1
            if patience == 0:
                break
    return test_model, best_val_result, best_test_result

In [11]:
import optuna
import gc

def search_hyperparams(dataset_name: str, method: str, base_hyperparams: dict, n_trials: int = 50, exp_cfg: dict | None = None) -> optuna.Study:

    def get_hyperparams(trial: optuna.Trial) -> dict:
        hyperparams = base_hyperparams.copy()
        hyperparams['method'] = method

        hyperparams['cand_size_exp'] = trial.suggest_int('cand_size_exp', low=1, high=8, step=1)
        hyperparams['cand_size'] = 2 ** hyperparams['cand_size_exp']

        if hyperparams['method'] == 'ans':
            hyperparams['gamma'] = trial.suggest_float('gamma', low=0.01, high=0.50, step=0.01)
            hyperparams['epsilon'] = trial.suggest_float('epsilon', low=0.01, high=1.00, step=0.01)
        elif hyperparams['method'] == 'shns':
            # hyperparams['alpha_step'] = trial.suggest_float('alpha_step', low=0.0001, high=0.0050, step=0.0001)
            hyperparams['alpha_step'] = 1e-5 * 2 ** trial.suggest_int('alpha_step_exp', low=0, high=9, step=1)
            hyperparams['beta'] = trial.suggest_float('beta', low=0.1, high=10.0, step=0.1)
        elif hyperparams['method'] == 'pure_shns':
            hyperparams['alpha_step'] = 1e-5 * 2 ** trial.suggest_int('alpha_step_exp', 0.0001, 0.01, step=0.0001)
            hyperparams['alpha_smart'] = float(hyperparams.get('alpha_start', 0.001))
            hyperparams['alpha_max'] = float(hyperparams.get('alpha_max', 0.5))
        elif hyperparams['method'] == 'ahns':
            hyperparams['alpha'] = trial.suggest_float('alpha', low=0.1, high=1.0, step=0.1)
            hyperparams['beta'] = trial.suggest_float('beta', low=0.1, high=1.0, step=0.1)
        elif hyperparams['method'] == 'dins':
            hyperparams['beta'] = trial.suggest_float('beta', low=0.1, high=10.0, step=0.1)
        elif hyperparams['method'] == 'dns_mn':
            hyperparams['m_ratio'] = trial.suggest_float('m_ratio', low=0.01, high=0.50, step=0.01)
        elif hyperparams['method'] == 'dns':
            pass
        else:
            raise ValueError(f'Unknown method: {hyperparams["method"]}')
        return hyperparams

    def objective(trial, dataset: Dataset):
        gc.collect()
        torch.cuda.empty_cache()
        hyperparams = get_hyperparams(trial)
        print(hyperparams)
        _, best_val_result, best_test_result = train(dataset, hyperparams, print_result=True)
        print(f'test: {best_test_result}')
        return best_val_result['recall_20']

    dataset = get_dataset(dataset_name, exp_cfg=exp_cfg)
    base_hyperparams = base_hyperparams.copy()
    
    study_name = f'{dataset_name}_dataset_{method}_layer_num_{base_hyperparams["layer_num"]}'
    sampler = optuna.samplers.TPESampler()
    study = optuna.create_study(
        study_name=study_name,
        direction='maximize',
        sampler=sampler
    )
    study.optimize(lambda trial: objective(trial, dataset), n_trials=n_trials)
    return study

/home/seyeon/lab/shns-notebook/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import copy
import math

def get_test_result(dataset_name: str, hyperparams: dict, n_trials: int = 10, exp_cfg: dict | None = None) -> dict:
    def is_number(x):
        return isinstance(x, (int, float)) and not (isinstance(x, float) and math.isnan(x))

    avg_best_test_result = None
    prev_good_result = None
    valid_trials = 0

    dataset = get_dataset(dataset_name, exp_cfg=exp_cfg)
    hyperparams = copy.deepcopy(hyperparams)

    for _ in range(n_trials):
        _, _, best_test_result = train(dataset, hyperparams, print_result=True)
        print(best_test_result)

        cur = best_test_result
        cur_recall = cur["recall_20"]
        prev_recall = None if prev_good_result is None else prev_good_result["recall_20"]

        is_outlier = (
            prev_good_result is not None
            and is_number(cur_recall)
            and is_number(prev_recall)
            and prev_recall > 0
            and cur_recall < 0.5 * prev_recall
        )

        if is_outlier:
            continue

        prev_good_result = cur
        if avg_best_test_result is None:
            avg_best_test_result = {k: float(v) for k, v in cur.items() if is_number(v)}
        else:
            for k, v in cur.items():
                if is_number(v):
                    avg_best_test_result[k] = avg_best_test_result.get(k, 0.0) + float(v)

        valid_trials += 1

    for k in list(avg_best_test_result.keys()):
        avg_best_test_result[k] /= valid_trials

    return avg_best_test_result

In [13]:
import os
import tempfile
import requests
from typing import List
import numpy as np
import copy
from dotenv import load_dotenv
load_dotenv()

import matplotlib.pyplot as plt
from optuna.visualization.matplotlib import plot_slice


NOTION_VERSION = "2025-09-03"

def save_to_notion(study, test_result: dict):
    token = os.getenv("NOTION_API_TOKEN")
    page_id = os.getenv("NOTION_PAGE_ID")

    ax = plot_slice(study)
    fig = ax[0].figure

    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    tmp_path = tmp.name
    tmp.close()

    fig.savefig(tmp_path, dpi=200)
    plt.close(fig)

    h_json = {
        "Authorization": f"Bearer {token}",
        "Notion-Version": NOTION_VERSION,
        "Content-Type": "application/json",
    }
    h_send = {
        "Authorization": f"Bearer {token}",
        "Notion-Version": NOTION_VERSION,
    }

    r = requests.post(
        "https://api.notion.com/v1/file_uploads",
        headers=h_json,
        json={"filename": "plot_slice.png", "content_type": "image/png"},
    )
    r.raise_for_status()
    upload_id = r.json()["id"]

    with open(tmp_path, "rb") as f:
        r = requests.post(
            f"https://api.notion.com/v1/file_uploads/{upload_id}/send",
            headers=h_send,
            files={"file": ("plot_slice.png", f, "image/png")},
        )
    r.raise_for_status()

    test_str = "\n".join(f"{k}: {v}" for k, v in test_result.items())
    best_str = "\n".join(f"{k}: {v}" for k, v in study.best_params.items())

    payload = {
        "children": [
            {"object": "block", "type": "divider", "divider": {}},
            {
                "object": "block",
                "type": "heading_3",
                "heading_3": {
                    "rich_text": [{"type": "text", "text": {"content": f"{study.study_name} (best={study.best_value})"}}]
                },
            },
            {
                "object": "block",
                "type": "paragraph",
                "paragraph": {"rich_text": [{"type": "text", "text": {"content": "best_params\n" + best_str}}]},
            },
            {
                "object": "block",
                "type": "paragraph",
                "paragraph": {"rich_text": [{"type": "text", "text": {"content": "test_result\n" + test_str}}]},
            },
            {
                "object": "block",
                "type": "image",
                "image": {"type": "file_upload", "file_upload": {"id": upload_id}},
            },
        ]
    }

    r = requests.patch(
        f"https://api.notion.com/v1/blocks/{page_id}/children",
        headers=h_json,
        json=payload,
    )
    r.raise_for_status()

def _mean_std_dict(dict_list: list[dict]) -> tuple[dict, dict]:
    # dict_list에 있는 공통 key들에 대해 평균/표준편차 dict를 만듦
    keys = dict_list[0].keys()
    mean_d, std_d = {}, {}
    for k in keys:
        vals = [d[k] for d in dict_list if k in d and isinstance(d[k], (int, float, np.floating))]
        if len(vals) == 0:
            continue
        mean_d[k] = float(np.mean(vals))
        std_d[k] = float(np.std(vals))
    return mean_d, std_d

def save_to_notion_grid(title: str, best_params: dict, best_val: dict, best_test: dict):
    # Optuna plot_slice 같은 건 grid search에선 불가능(Study가 없음). 그래서 텍스트로만 결과 저장
    
    token = os.getenv("NOTION_API_TOKEN")
    page_id = os.getenv("NOTION_PAGE_ID")

    best_params_str = "\n".join(f"{k}: {v}" for k, v in best_params.items())
    best_val_str = "\n".join(f"{k}: {v}" for k, v in best_val.items())
    best_test_str = "\n".join(f"{k}: {v}" for k, v in best_test.items())

    payload = {
        "children": [
            {"object": "block", "type": "divider", "divider": {}},
            {
                "object": "block",
                "type": "heading_3",
                "heading_3": {
                    "rich_text": [{"type": "text", "text": {"content": title}}]
                },
            },
            {
                "object": "block",
                "type": "paragraph",
                "paragraph": {"rich_text": [{"type": "text", "text": {"content": "best_params\n" + best_params_str}}]},
            },
            {
                "object": "block",
                "type": "paragraph",
                "paragraph": {"rich_text": [{"type": "text", "text": {"content": "best_val\n" + best_val_str}}]},
            },
            {
                "object": "block",
                "type": "paragraph",
                "paragraph": {"rich_text": [{"type": "text", "text": {"content": "best_test\n" + best_test_str}}]},
            },
        ]
    }

    h_json = {
        "Authorization": f"Bearer {token}",
        "Notion-Version": NOTION_VERSION,
        "Content-Type": "application/json",
    }

    r = requests.patch(
        f"https://api.notion.com/v1/blocks/{page_id}/children",
        headers=h_json,
        json=payload,
    )
    r.raise_for_status()

def grid_search_cand_alpha_step(
    dataset_name: str,
    method: str,
    base_hyperparams: dict,
    exp_cfg: dict | None,
    cand_size_exp_list: List[int],
    alpha_step_exp_list: List[int],
    alpha_step_base: float = 1e-5,
    alpha_start: float = 0.001,
    alpha_max: float = 0.5,
    repeat: int = 5,
):
    dataset = get_dataset(dataset_name, exp_cfg=exp_cfg)

    # alpha grid 만들기 (0.01 단위)
    # alpha_max 포함되게 만들기 위해 작은 epsilon 추가
    alpha_step_grid = [float(alpha_step_base * (2 ** int(k))) for k in alpha_step_exp_list]

    best_score = -1e18
    best_params = None
    best_val = None
    best_test = None

    for cand_exp in cand_size_exp_list:
        cand_size = 2 ** int(cand_exp)

        for k, alpha_step in zip(alpha_step_exp_list, alpha_step_grid):
            # 조합별 결과 누적
            val_list = []
            test_list = []
            recall20_list = []

            for r in range(repeat):
                hp = copy.deepcopy(base_hyperparams)
                hp["method"] = method
                hp["cand_size"] = int(cand_size)

                if method == "pure_shns":
                    hp["alpha_step"] = float(alpha_step)
                    hp["alpha_start"] = float(alpha_start)
                    hp["alpha_max"] = float(alpha_max)
                else:
                    # 다른 method도 grid로 할 거면 여기 확장하면 됨
                    raise ValueError(f"grid_search_cand_alpha_step currently supports pure_shns only, got {method}")

                # 학습
                _, val_res, test_res = train(dataset, hp, print_result=False)

                val_list.append(val_res)
                test_list.append(test_res)
                recall20_list.append(val_res["recall_20"])

            avg_score = float(np.mean(recall20_list))
            std_score = float(np.std(recall20_list))

            print(f"cand_size={cand_size}, alpha_step={alpha_step:.6f} => avg val recall_20={avg_score:.10f}")

            val_mean, val_std = _mean_std_dict(val_list)
            test_mean, test_std = _mean_std_dict(test_list)

            if avg_score > best_score:
                best_score = avg_score
                best_params = {
                    "cand_size": int(cand_size),
                    "cand_size_exp": int(cand_exp),
                    "alpha_step": float(alpha_step),
                    "alpha_step_exp": int(k),
                    "alpha_step_base": float(alpha_step_base),
                    "alpha_start": float(alpha_start),
                    "alpha_max": float(alpha_max),
                    "repeat": int(repeat),
                }
                
                # 노션에 “평균” 기준으로 저장 (원하면 std도 같이)
                best_val = {
                    **{f"mean_{kk}": vv for kk, vv in val_mean.items()},
                    **{f"std_{kk}": vv for kk, vv in val_std.items()},
                    "selected_by": "avg(val_recall_20)",
                    "avg_val_recall_20": avg_score,
                    "std_val_recall_20": std_score,
                }
                best_test = {
                    **{f"mean_{kk}": vv for kk, vv in test_mean.items()},
                    **{f"std_{kk}": vv for kk, vv in test_std.items()},
                }

    return best_params, best_val, best_test

In [14]:
methods = ['pure_shns'] #'ahns', 'dins', 'ans', 'dns', 'dns_mn'
layer_nums = [0]
dataset_names = ['tool']

cand_size_exp_list = list(range(1, 9))
alpha_step_exp_list = list(range(1, 13))

for dataset_name in dataset_names:
    for method in methods:
        for layer_num in layer_nums:
            exp_cfg = None
            
            print(f"\n==== dataset={dataset_name}, method={method}, layer_num={layer_num} ====\n")
            
            base_hyperparams= {
                'layer_num': layer_num,
                'reg': 1e-5,
                'batch_size': 512,
                'emb_size': 32,
                'epochs': 100,
                'patience': 10,
            }

            best_params, best_val, best_test = grid_search_cand_alpha_step(
                dataset_name=dataset_name,
                method=method,
                base_hyperparams=base_hyperparams,
                exp_cfg=exp_cfg,
                cand_size_exp_list=cand_size_exp_list,
                alpha_step_exp_list=alpha_step_exp_list,
                alpha_step_base=1e-5,
                alpha_start=0.001,
                alpha_max=0.5,
                repeat=5,
            )

            print("\n[BEST GRID PARAMS]")
            print(best_params)
            print("[BEST VAL]")
            print(best_val)
            print("[BEST TEST]")
            print(best_test)

            # (선택) best params로 test를 더 많이 돌려 평균낼 수도 있음
            best_hyperparams = copy.deepcopy(base_hyperparams)
            best_hyperparams["method"] = method
            best_hyperparams["cand_size"] = int(best_params["cand_size"])
            best_hyperparams["alpha_step"] = float(best_params["alpha_step"])
            best_hyperparams["alpha_start"] = float(best_params["alpha_start"])
            best_hyperparams["alpha_max"] = float(best_params["alpha_max"])

            # Notion 저장 grid용 함수로
            title = f"{dataset_name} / {method} / layer={layer_num} / best_avg_val={best_val['avg_val_recall_20']}"
            save_to_notion_grid(title, best_params, best_val, best_test)


==== dataset=tool, method=pure_shns, layer_num=0 ====



FileNotFoundError: [Errno 2] No such file or directory: './SHNS_data/tool/train.txt'